# Variational Autoencoder (VAE) for Face Generation on FFHQ

This notebook implements a Variational Autoencoder (VAE) to learn and reconstruct human faces using the Flickr-Faces-HQ (FFHQ) dataset.

In [1]:
import kagglehub
arnaud58_flickrfaceshq_dataset_ffhq_path = kagglehub.dataset_download('arnaud58/flickrfaceshq-dataset-ffhq')

print('Data source import complete.')

Data source import complete.


In [2]:
import glob
import os
from pathlib import Path

import torch
import torch.nn.functional as F
from torchdiffeq import odeint
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.transforms import v2
from einops import rearrange

In [3]:
torch.set_float32_matmul_precision('high')

class FFHQDataset(Dataset):
  def __init__(self, root_dir, transform=None):
    self.root_dir = root_dir
    self.transform = transform
    self.image_paths = []
    for ext in ('*.png', '*.jpg', '*.jpeg'):
      self.image_paths.extend(glob.glob(os.path.join(root_dir, ext)))

    self.image_paths.sort()
    print(f'Found {len(self.image_paths)} total images in {root_dir}')

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    img_path = self.image_paths[idx]
    image = Image.open(img_path).convert('RGB')

    if self.transform:
      image = self.transform(image)
    else:
      default_transform = v2.Compose([
        v2.Resize((256, 256)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
      ])
      image = default_transform(image)

    return image

full_dataset = FFHQDataset(root_dir=arnaud58_flickrfaceshq_dataset_ffhq_path, transform=None)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f'Training set size: {len(train_dataset)}')
print(f'Testing set size: {len(test_dataset)}')

if len(train_dataset) > 0:
  images = next(iter(train_loader))
  print(f'Batch shape: {images.shape}')

Found 52001 total images in /teamspace/studios/this_studio/.cache/kagglehub/datasets/arnaud58/flickrfaceshq-dataset-ffhq/versions/1
Training set size: 41600
Testing set size: 10401
Batch shape: torch.Size([64, 3, 256, 256])


In [4]:
class SelfAttention(nn.Module):
  def __init__(self, in_channels, num_heads=4):
    super().__init__()
    self.mha = nn.MultiheadAttention(embed_dim=in_channels, num_heads=num_heads, batch_first=True)
    self.ln = nn.LayerNorm(in_channels)

  def forward(self, x):
    B, C, H, W = x.shape
    residual = x
    # Reshape (B, C, H, W) to (B, H*W, C)
    x_flat = x.permute(0, 2, 3, 1).contiguous().view(B, H * W, C)
    x_normed = self.ln(x_flat)
    attn, _ = self.mha(x_normed, x_normed, x_normed)
    out = attn.view(B, H, W, C).permute(0, 3, 1, 2).contiguous()
    return out + residual

class Encoder(nn.Module):
  def __init__(self, latent_channels=4, in_channels=3):
    super().__init__()
    self.conv = nn.Sequential(
      nn.Conv2d(in_channels, 64, 4, stride=2, padding=1), # 3x256x256 -> 64x128x128
      nn.ReLU(inplace=True),
      nn.Conv2d(64, 128, 4, stride=2, padding=1), # 64x128x128 -> 128x64x64
      nn.ReLU(inplace=True),
      nn.Conv2d(128, 256, 4, stride=2, padding=1), # 128x64x64 -> 256x32x32
      nn.ReLU(inplace=True),
    )
    self.attn = SelfAttention(in_channels=256, num_heads=4)

    self.conv_mu = nn.Conv2d(256, latent_channels, 1)
    self.conv_logvar = nn.Conv2d(256, latent_channels, 1)

  def forward(self, x):
    conv_output = self.conv(x)
    attn_output = self.attn(conv_output)
    mu = self.conv_mu(attn_output)
    logvar = self.conv_logvar(attn_output)
    return mu, logvar

class Decoder(nn.Module):
  def __init__(self, latent_channels=4, out_channels=3):
    super().__init__()
    self.initial_conv = nn.Sequential(
      nn.Conv2d(latent_channels, 256, 3, padding=1),
      nn.ReLU(inplace=True)
    )
    self.attn = SelfAttention(in_channels=256, num_heads=4)

    self.deconv = nn.Sequential(
      # 256x32x32 -> 128x64x64
      nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
      nn.ReLU(inplace=True),

      # 128x64x64 -> 64x128x128
      nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
      nn.ReLU(inplace=True),

      # 64x128x128 -> 3x256x256
      nn.ConvTranspose2d(64, out_channels, 4, stride=2, padding=1),
      nn.Tanh(),
    )

  def forward(self, z):
    x = self.initial_conv(z)
    x = self.attn(x)
    return self.deconv(x)

class VAE(nn.Module):
  def __init__(self, image_channels=3, latent_channels=4):
    super().__init__()
    self.encoder = Encoder(latent_channels, image_channels)
    self.decoder = Decoder(latent_channels, image_channels)

  def reparam(self, mu, logvar):
    """
    reparameterization trick to keep gradients
    """
    std = torch.exp(logvar * 0.5)
    elipson = torch.randn_like(std)
    return mu + elipson * std

  def forward(self, x):
    mu, logvar = self.encoder(x)
    z = self.reparam(mu, logvar)
    return self.decoder(z), mu, logvar


In [5]:
def vae_loss(x_out, x, mu, logvar, percep_loss_fn, beta=1e-4):
  percep_loss = percep_loss_fn(x_out, x).mean() * 0.6
  mse_term = F.mse_loss(x_out, x) * 0.2
  L1_term = F.l1_loss(x_out, x) * 0.8
  kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
  kl_term = kl_loss * beta
  return mse_term + L1_term + kl_term + percep_loss, mse_term, L1_term, kl_term, percep_loss

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VAE_CHECKPOINT_PATH = '../outputs/vae_checkpoint.pth'
vae = VAE(latent_channels=8).to(device)
ck = torch.load(VAE_CHECKPOINT_PATH, map_location='cpu')
if isinstance(ck, dict) and any(k in ck for k in ('state_dict','model_state_dict','model')):
    state = ck.get('state_dict') or ck.get('model_state_dict') or ck.get('model')
else:
    state = ck
clean = {}
for k, v in state.items():
    nk = k.replace('module.', '').replace('_orig_mod.', '')
    clean[nk] = v
res = vae.load_state_dict(clean, strict=False)
print('missing', res.missing_keys)
print('unexpected', res.unexpected_keys)
vae.to(device); vae.eval()
for p in vae.parameters(): p.requires_grad = False
print('VAE loaded')


missing []
unexpected []
VAE loaded


In [7]:
class CachedLatentDataset(Dataset):
  def __init__(self, latents):
    self.latents = latents

  def __len__(self):
    return self.latents.shape[0]

  def __getitem__(self, index):
    return self.latents[index]


def encode_dataset_to_latents(image_dataset, vae, device, batch_size=64, num_workers=0, sample_posterior=True):
  # DataLoader tuned for throughput
  loader_kwargs = {
    'batch_size': batch_size,
    'shuffle': False,
    'num_workers': num_workers,
    'pin_memory': (device.type == 'cuda'),
    'persistent_workers': (num_workers > 0),
  }
  if num_workers > 0:
    loader_kwargs['prefetch_factor'] = 2
  image_loader = DataLoader(image_dataset, **loader_kwargs)

  latent_batches = []
  vae.to(device)
  vae.eval()
  with torch.no_grad():
    for image_batch in image_loader:
      image_batch = image_batch.to(device, non_blocking=True)
      mu, logvar = vae.encoder(image_batch)
      latent_batch = vae.reparam(mu, logvar) if sample_posterior else mu
      latent_batches.append(latent_batch.cpu())

  return torch.cat(latent_batches, dim=0) if len(latent_batches) > 0 else torch.empty(0)


def build_or_load_latent_cache(
  train_dataset,
  test_dataset,
  vae,
  device,
  cache_dir='outputs/latent_cache',
  batch_size=64,
  num_workers=0,
  sample_posterior=True,
  force_rebuild=False,
):
  cache_path = Path(cache_dir)
  cache_path.mkdir(parents=True, exist_ok=True)

  mode = 'posterior' if sample_posterior else 'mu'
  train_cache_file = cache_path / f'train_latents_{mode}.pt'
  test_cache_file = cache_path / f'test_latents_{mode}.pt'

  if force_rebuild or (not train_cache_file.exists()) or (not test_cache_file.exists()):
    print('Building latent cache...')
    train_latents = encode_dataset_to_latents(
      train_dataset,
      vae,
      device,
      batch_size=batch_size,
      num_workers=num_workers,
      sample_posterior=sample_posterior,
    )
    test_latents = encode_dataset_to_latents(
      test_dataset,
      vae,
      device,
      batch_size=batch_size,
      num_workers=num_workers,
      sample_posterior=sample_posterior,
    )
    torch.save(train_latents, train_cache_file)
    torch.save(test_latents, test_cache_file)
  else:
    print('Loading latent cache from disk...')

  train_latents = torch.load(train_cache_file, map_location='cpu')
  test_latents = torch.load(test_cache_file, map_location='cpu')

  return train_latents, test_latents


def build_cached_latent_loaders(train_latents, test_latents, batch_size=64, num_workers=0):
  train_latent_dataset = CachedLatentDataset(train_latents)
  test_latent_dataset = CachedLatentDataset(test_latents)

  train_kwargs = {
    'batch_size': batch_size,
    'shuffle': True,
    'num_workers': num_workers,
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': (num_workers > 0),
  }
  if num_workers > 0:
    train_kwargs['prefetch_factor'] = 2
  train_latent_loader = DataLoader(train_latent_dataset, **train_kwargs)

  test_kwargs = {
    'batch_size': batch_size,
    'shuffle': False,
    'num_workers': num_workers,
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': (num_workers > 0),
  }
  if num_workers > 0:
    test_kwargs['prefetch_factor'] = 2
  test_latent_loader = DataLoader(test_latent_dataset, **test_kwargs)

  return train_latent_loader, test_latent_loader


In [8]:
def sinusoidal_time_embedding(t, embedding_dim, max_period=10_000):
  """
  Encode timestep values into sinusoidal embeddings using sine and cosine functions
  at different frequencies. Used in diffusion models for time-aware conditioning.

  Args:
    t: Timestep values in [0, 1]. Shape: [B] or [B, 1]
    embedding_dim: Output embedding dimension
    max_period: Period of the lowest frequency (default 10,000)

  Returns:
    Sinusoidal embedding tensor of shape [B, embedding_dim]
  """
  if t.dim() == 2:
    t = t.squeeze(-1)

  half_dim = embedding_dim // 2
  freqs = torch.exp(
    -torch.log(torch.tensor(float(max_period), device=t.device))
    * torch.arange(half_dim, device=t.device, dtype=torch.float32)
    / max(half_dim - 1, 1)
  )

  angles = t.float().unsqueeze(1) * freqs.unsqueeze(0)
  emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)

  if embedding_dim % 2 == 1:
    emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)

  return emb


class TimeEmbeddingMLP(nn.Module):
  """SiLU-based MLP to project time embeddings to model dimension.

  Example:
    sin_emb = sinusoidal_time_embedding(t, 128)
    proj = TimeEmbeddingMLP(input_dim=128, out_dim=384)
    time_feat = proj(sin_emb)  # shape [B, 384]
  """
  def __init__(self, input_dim=128, out_dim=384, hidden_dim=None):
    super().__init__()
    if hidden_dim is None:
      hidden_dim = max(out_dim, input_dim * 2)

    self.net = nn.Sequential(
      nn.Linear(input_dim, hidden_dim),
      nn.SiLU(),
      nn.Linear(hidden_dim, out_dim),
    )

  def forward(self, x):
    return self.net(x)


def projected_time_embedding(t, sin_dim=128, out_dim=384, max_period=10_000, mlp=None, device=None):
  """Helper: compute sinusoidal embedding then project via provided MLP (or a default one).

  Returns tensor shape [B, out_dim].
  """
  if device is None:
    device = t.device

  sin_emb = sinusoidal_time_embedding(t.to(device), sin_dim, max_period)
  if mlp is None:
    mlp = TimeEmbeddingMLP(input_dim=sin_dim, out_dim=out_dim).to(device)

  return mlp(sin_emb)


In [9]:
class PatchEmbedConv(nn.Module):
  """Conv2d-based patch embedder for latent grids using einops for reshaping.

  Converts `x` of shape [B, in_ch, H, W] into tokens [B, N, embed_dim]
  by applying a Conv2d projection with `kernel_size=patch_size` and
  flattening the spatial dims. No normalization applied here.
  """
  def __init__(self, in_ch=8, embed_dim=384, patch_size=4, stride=2):
    super().__init__()
    self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=stride)

  def forward(self, x):
    # x: [B, in_ch, H, W]
    x = self.proj(x) # [B, embed_dim, H', W']
    x = rearrange(x, 'b d h w -> b (h w) d')
    return x


In [10]:
class QKVProjector(nn.Module):
  """Project input tokens to Q, K, V and return shaped tensors per-head.

  Returns q,k,v each with shape [B, N, num_heads, head_dim].
  """
  def __init__(self, dim, num_heads):
    super().__init__()
    assert dim % num_heads == 0
    self.dim = dim
    self.num_heads = num_heads
    self.head_dim = dim // num_heads
    self.qkv = nn.Linear(dim, dim * 3)

  def forward(self, x):
    # x: [B, N, D]
    B, N, D = x.shape
    qkv = self.qkv(x)  # [B, N, 3*D]
    qkv = qkv.view(B, N, 3, self.num_heads, self.head_dim)
    q = qkv[:, :, 0]
    k = qkv[:, :, 1]
    v = qkv[:, :, 2]
    return q, k, v

class MultiHeadSelfAttentionPlain(nn.Module):
  """Standard multi-head self-attention (no RoPE).

  Inputs:
    x: [B, N, D]
  Output:
    [B, N, D]
  """
  def __init__(self, dim, num_heads=8, dropout=0.0):
    super().__init__()
    assert dim % num_heads == 0
    self.dim = dim
    self.num_heads = num_heads
    self.head_dim = dim // num_heads

    self.qkv_proj = QKVProjector(dim, num_heads)
    self.q_norm = nn.LayerNorm(self.head_dim)
    self.k_norm = nn.LayerNorm(self.head_dim)
    self.out_proj = nn.Linear(dim, dim)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, attn_mask=None):
    B, N, D = x.shape
    q, k, v = self.qkv_proj(x)  # each: [B, N, num_heads, head_dim]

    # [B, heads, N, head_dim]
    q = rearrange(q, 'b n h d -> b h n d')
    k = rearrange(k, 'b n h d -> b h n d')
    v = rearrange(v, 'b n h d -> b h n d')
    
    q = self.q_norm(q)
    k = self.k_norm(k)

    scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
    if attn_mask is not None:
      scores = scores + attn_mask
    attn = torch.softmax(scores, dim=-1)
    attn = self.dropout(attn)

    out = torch.matmul(attn, v)  # [B, heads, N, head_dim]
    out = rearrange(out, 'b h n d -> b n (h d)')

    return self.out_proj(out)


class TransformerBlockPlain(nn.Module):
  """Pre-norm transformer block using the plain attention above.

  Usage:
    block = TransformerBlockPlain(dim=384, num_heads=8)
    out = block(tokens)  # tokens: [B, N, D]
  """
  def __init__(self, dim, num_heads=8, mlp_ratio=4.0, dropout=0.0):
    super().__init__()
    self.norm1 = nn.LayerNorm(dim)
    self.attn = MultiHeadSelfAttentionPlain(dim, num_heads=num_heads, dropout=dropout)
    self.norm2 = nn.LayerNorm(dim)
    hidden_dim = int(dim * mlp_ratio)
    self.mlp = nn.Sequential(
      nn.Linear(dim, hidden_dim),
      nn.SiLU(),
      nn.Linear(hidden_dim, dim),
      nn.Dropout(dropout),
    )

  def forward(self, x):
    x = x + self.attn(self.norm1(x))
    x = x + self.mlp(self.norm2(x))
    return x


In [11]:
class LearnedPositionalEmbedding(nn.Module):
  """Learned 1D absolute positional embeddings.

  Store a parameter of shape [1, max_positions, dim]. For an input
  token tensor of shape [B, N, D], return the first N positions and
  broadcast for addition.
  """
  def __init__(self, max_positions=1024, dim=384):
    super().__init__()
    self.max_positions = max_positions
    self.dim = dim
    self.pos = nn.Parameter(nn.init.normal_(torch.empty(1, max_positions, dim), std=0.02))

  def forward(self, x):
    # x: [B, N, D]
    N = x.shape[1]
    if N > self.max_positions:
      raise ValueError(f"Requested {N} positions but max is {self.max_positions}")
    return self.pos[:, :N, :].to(x.device)


In [12]:
class ViT(nn.Module):
    """Vision Transformer for latent space modeling.
    
    Architecture:
        - Patch embedding via Conv2d
        - Add learned positional embeddings and timestamp embeddings
        - Stack of transformer blocks with plain multi-head self-attention
    """
    def __init__(
        self,
        latent_w=32,
        latent_h=32,
        patch_size=4,
        stride=2,
        in_channels=8,
        embed_dim=384,
        num_heads=8,
        num_layers=12,
        mlp_ratio=4.0,
        dropout=0.0,
    ):
        super().__init__()

        self.latent_w = latent_w
        self.latent_h = latent_h
        self.patch_size = patch_size
        self.stride = stride
        self.in_channels = in_channels
        self.embed_dim = embed_dim
        
        def _tokens_for_size(size, kernel, stride, padding=0, dilation=1):
            return (size + 2 * padding - dilation * (kernel - 1) - 1) // stride + 1

        h_tokens = _tokens_for_size(latent_h, patch_size, stride)
        w_tokens = _tokens_for_size(latent_w, patch_size, stride)
        num_tokens = h_tokens * w_tokens
        self.num_tokens = int(num_tokens)

        self.patch_embed = PatchEmbedConv(in_channels, embed_dim, patch_size, stride)
        self.pos_embed = LearnedPositionalEmbedding(max_positions=self.num_tokens, dim=embed_dim)
        self.time_mlp = TimeEmbeddingMLP(input_dim=128, out_dim=embed_dim)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlockPlain(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.spatial_project = nn.ConvTranspose2d(
            in_channels=embed_dim,
            out_channels=in_channels,
            kernel_size=patch_size,
            stride=stride,
        )

    def forward(self, x, t):
        """x: [B, in_channels, latent_w, latent_h]
           t: [B] or [B, 1] with values in [0, 1]
        """

        B, C, H, W = x.shape
        tokens = self.patch_embed(x)  # [B, N, embed_dim]

        if tokens.shape[1] != self.num_tokens:
            raise ValueError(f"Computed tokens ({tokens.shape[1]}) != expected ({self.num_tokens}).\n"
                             "Ensure input spatial size matches ViT latent_w/latent_h and patch settings.")

        pos_emb = self.pos_embed(tokens)  # [1, N, embed_dim]
        time_emb = projected_time_embedding(t, mlp=self.time_mlp, device=x.device).unsqueeze(1)  # [B, 1, embed_dim]
        x = tokens + pos_emb + time_emb  # [B, N, embed_dim]

        for block in self.transformer_blocks:
            x = block(x)  # [B, N, embed_dim]
            
        def _tokens_for_size(size, kernel, stride, padding=0, dilation=1):
            return (size + 2 * padding - dilation * (kernel - 1) - 1) // stride + 1
            
        B, N, D = x.shape
        h_tokens = _tokens_for_size(self.latent_h, self.patch_size, self.stride)
        w_tokens = _tokens_for_size(self.latent_w, self.patch_size, self.stride)
        
        x_map = rearrange(x, 'b (h w) d -> b d h w', h=h_tokens, w=w_tokens)

        out = self.spatial_project(x_map)
        return out

In [ ]:
from sympy import rem
import os

import lightning as pl
from IPython.display import display
from lightning.pytorch.callbacks import ModelCheckpoint
from torchvision.transforms.functional import to_pil_image
from torchvision.utils import make_grid

class LatentFlowMatchingModule(pl.LightningModule):
  def __init__(self, latent_shape=(8, 32, 32), lr=2e-4, weight_decay=1e-2, num_flow_steps=50):
    super().__init__()
    self.save_hyperparameters()
    self.latent_shape = latent_shape
    self.lr = lr
    self.weight_decay = weight_decay
    self.num_flow_steps = num_flow_steps
    self.flow = ViT(
      latent_w=latent_shape[2],
      latent_h=latent_shape[1],
      patch_size=4,
      stride=2,
      in_channels=latent_shape[0],
      embed_dim=768,
      num_heads=10,
      num_layers=10,
      mlp_ratio=4.0,
      dropout=0.0,
    )

  def flow_matching_loss(self, latents):
    batch_size = latents.shape[0]
    x1 = latents
    x0 = torch.randn_like(x1)
    t = torch.rand(batch_size, device=latents.device)
    t_view = t.view(batch_size, 1, 1, 1)
    x_t = (1.0 - t_view) * x0 + t_view * x1
    target_velocity = x1 - x0
    pred_velocity = self.flow(x_t, t)
    loss = F.mse_loss(pred_velocity, target_velocity)
    return loss

  def training_step(self, batch, batch_idx):
    latents = batch.float()
    loss = self.flow_matching_loss(latents)
    self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, batch_size=latents.shape[0])
    return loss

  def validation_step(self, batch, batch_idx):
    latents = batch.float()
    loss = self.flow_matching_loss(latents)
    self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=latents.shape[0])
    return loss

  def configure_optimizers(self):
    optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
    total_epochs = self.trainer.max_epochs if self.trainer is not None else 80
    warmup_epochs = 5
    remaining_epochs = max(total_epochs - warmup_epochs, 1)
    cosine_period = remaining_epochs // 3

    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cosine = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=cosine_period, T_mult=1, eta_min=self.lr * 1e-3)
    
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

    return {
      'optimizer': optimizer,
      'lr_scheduler': {
        'scheduler': scheduler,
        'interval': 'epoch',
      },
    }

  @torch.no_grad()
  def sample_latents(self, num_samples=8, steps=None, solver="dopri5"):
    if steps is None:
      steps = self.num_flow_steps
    sample_shape = (num_samples, *self.latent_shape)
    x0 = torch.randn(sample_shape, device=self.device)

    def ode_func(t, x):
      t_batch = t.expand(x.shape[0]).to(x.device)
      return self.flow(x, t_batch)

    t_eval = torch.linspace(0.0, 1.0, steps + 1, device=self.device)
    traj = odeint(ode_func, x0, t_eval, method=solver)
    return traj[-1]


class EpochImageLogger(pl.Callback):
  def __init__(self, vae, num_samples=8, every_n_epochs=1, out_dir='outputs/flow_matching/samples'):
    super().__init__()
    self.vae = vae
    self.num_samples = num_samples
    self.every_n_epochs = every_n_epochs
    self.out_dir = out_dir

  def on_validation_epoch_end(self, trainer, pl_module):
    if (trainer.current_epoch + 1) % self.every_n_epochs != 0:
      return
    if not trainer.is_global_zero:
      return
    pl_module.eval()
    self.vae.to(pl_module.device)
    self.vae.eval()
    with torch.no_grad():
      latents = pl_module.sample_latents(num_samples=self.num_samples)
      images = self.vae.decoder(latents)
      images = (images.clamp(-1, 1) + 1) / 2
      nrow = min(4, self.num_samples)
      grid = make_grid(images, nrow=nrow)
      os.makedirs(self.out_dir, exist_ok=True)
      path = os.path.join(self.out_dir, f'epoch_{trainer.current_epoch:04d}.png')
      to_pil_image(grid.cpu()).save(path)
      display(to_pil_image(grid.cpu()))
      if trainer.logger is not None:
        experiment = getattr(trainer.logger, 'experiment', None)
        if experiment is not None and hasattr(experiment, 'add_image'):
          experiment.add_image('samples', grid, trainer.current_epoch)
    pl_module.train()


latent_train_latents, latent_val_latents = build_or_load_latent_cache(
  train_dataset=train_dataset,
  test_dataset=test_dataset,
  vae=vae,
  device=device,
  cache_dir='outputs/latent_cache',
  batch_size=256,
  num_workers=8,
  sample_posterior=False,
  force_rebuild=False,
)

latent_train_loader, latent_val_loader = build_cached_latent_loaders(
  latent_train_latents,
  latent_val_latents,
  batch_size=256,
  num_workers=8,
)

flow_module = LatentFlowMatchingModule(latent_shape=(8, 32, 32), lr=8e-4, weight_decay=1e-2, num_flow_steps=50)

checkpoint_callback = ModelCheckpoint(
  dirpath='outputs/flow_matching/checkpoints',
  filename='epoch_{epoch:04d}',
  save_last=True,
  save_top_k=-1,
  every_n_epochs=1,
)

trainer = pl.Trainer(
  max_epochs=80,
  accelerator='auto',
  devices='auto',
  precision='16-mixed' if torch.cuda.is_available() else '32-true',
  gradient_clip_val=1.0,
  log_every_n_steps=10,
  default_root_dir='outputs/flow_matching',
  callbacks=[
    EpochImageLogger(vae=vae, num_samples=8, every_n_epochs=1),
    checkpoint_callback,
  ],
)

trainer.fit(flow_module, train_dataloaders=latent_train_loader, val_dataloaders=latent_val_loader)
